## Topic: ContextualCompressionRetriever

### Agenda
- 1. Introduction of ContextualCompressionRetriever

- 2. Syntax of ContextualCompressionRetriever

- 3. Practical Examples of ContextualCompressionRetriever

- 4. Complete Summary of ContextualCompressionRetriever

### 1. Introduction of ContextualCompressionRetriever

- Definition:
    - The Contextual Compression Retriever in LangChain is an advanced retriever that improves retrieval quality by compressing documents after retrieval — keeping only the relevant content based on the user's query.


    - ContextualCompressionRetriever is a wrapper retriever in LangChain.

    - It first retrieves documents using a normal/base retriever, then applies a compressor to remove irrelevant content, filter weak documents, or re-rank results.

    

In [ ]:
""" 
    ContextualCompressionRetriever — Complete Visual Representation
    =====================================================================

┌─────────────────────────────────────────────────────────────────────┐
│                 CONTEXTUAL COMPRESSION RETRIEVER                    │
└─────────────────────────────────────────────────────────────────────┘

                              USER
                               │
                               │
                               ▼
                    ┌───────────────────┐
                    │      Query        │
                    │                   │
                    │ "What is CNN?"    │
                    └─────────┬─────────┘
                              │
                              ▼
                ┌──────────────────────────┐
                │      BASE RETRIEVER      │
                │                          │
                │ Similarity / MMR /       │
                │ Vector Store Retriever   │
                └────────────┬─────────────┘
                             │
                             │ Retrieves
                             ▼
              ┌───────────────────────────────┐
              │       Retrieved Documents     │
              │                               │
              │ Document 1 ── Relevant        │
              │ Document 2 ── Partly relevant │
              │ Document 3 ── Irrelevant      │
              │ Document 4 ── Relevant        │
              │ Document 5 ── Noise           │
              └───────────────┬───────────────┘
                              │
                              ▼
                 ┌────────────────────────┐
                 │   DOCUMENT COMPRESSOR  │
                 │                        │
                 │   Analyze documents    │
                 │         │              │
                 │    ┌────┴────┐         │
                 │    ▼         ▼         │
                 │ Relevant   Irrelevant │
                 │ Content     Content    │
                 └────┬─────────┬─────────┘
                      │         │
                      │         ✕ Remove
                      │
                      ▼
             ┌──────────────────────────┐
             │   COMPRESSED CONTEXT     │
             │                          │
             │ Relevant information     │
             │ only                     │
             └────────────┬─────────────┘
                          │
                          ▼
                  ┌───────────────┐
                  │      LLM      │
                  │               │
                  │ Generate      │
                  │ Answer        │
                  └───────┬───────┘
                          │
                          ▼
                     ┌─────────┐
                     │ Answer  │
                     └─────────┘

# Core Idea
    - A normal retriever may retrieve useful documents, but each document may include lots of unnecessary text.

"""

In [ ]:
"""
        HOW  CONTEXTUAL COMPRESSION RETRIEVER INTERNAL FLOW 
        =====================================================

┌─────────────────────────────────────────────────────────────┐
│      CONTEXTUAL COMPRESSION RETRIEVER INTERNAL FLOW         │
│                                                             │
│  1. RECEIVE USER QUERY                                      │
│     "What is the refund policy?"                            │
│                                                             │
│  2. BASE RETRIEVER FETCHES DOCUMENTS                        │
│     Vector / BM25 / MMR / MultiQuery retrieves top-k docs   │
│                                                             │
│     Example:                                                │
│     Doc 1 → Full refund policy page                         │
│     Doc 2 → Shipping and returns guide                      │
│     Doc 3 → Customer support FAQ                            │
│                                                             │
│  3. COMPRESSOR PROCESSES EACH DOCUMENT                      │
│                                                             │
│     A. Filter mode:                                         │
│        Keep relevant docs, remove irrelevant docs           │
│                                                             │
│     B. Extract mode:                                        │
│        Keep only relevant sentences/passages                │
│                                                             │
│     C. Re-rank mode:                                        │
│        Score docs again and return best documents           │
│                                                             │
│  4. BUILD COMPRESSED DOCUMENTS                              │
│     Document(                                               │
│       page_content="Refund available within 30 days...",    │
│       metadata={"source": "refund_policy.pdf", "page": 2}   │
│     )                                                       │
│                                                             │
│  5. RETURN CLEAN List[Document]                             │
│     Less noise, fewer tokens, more focused context          │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""- Why do We need Contextual Compression Retriever


- Retried Document (by a traditional retriever):
====================================================
    The Grand Canyon is a famous natural site
    Photosynthesis is how plants convert light into energy
    many tourist visit every year.

  - User query:
    - What is photosynthesis?

- Here the Problem of User Query (for traditional retriever):
====================================================
    - 1. The retriever return the entire paragraph
    - 2. only one sentence is actually relevant to the query.
    - 3. The rest is irrelevant noise that wastes context window and may confuse the LLM.


- for contextual compression retriever:
====================================================
    - response: Return only the relevant part 
    - eg: Photosynthesis is how plants convert light into energy

    - How it works?
        - 1. Base Retriever (eg. FAISS, Chroma) retrieves N documents.
        - 2. A Compressor (usually an LLM) is applied to each document.
        - 3. The Compressor keeps only the parts relevant to the query.
        - 4. Irrelevant content is discarded.


- Why to use:
    - Your documents are long and contain mixed information.
    - You want to reduce context length for LLMs.
    - You need to improve answer accuracy in RAG pipeline.
    
 """

### 2. Syntax of ContextualCompressionRetriever

In [ ]:
""" 
"""
        - General Syntax
        ===================

from langchain.retrievers import ContextualCompressionRetriever

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

docs = compression_retriever.invoke("your question")



- Where,
==================================
    # 1. Base retriever returns full documents/chunks
        base_retriever = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 2}
        )
    
    # 2. Extract only query-relevant parts using LLM
    from langchain.retrievers.document_compressors import LLMChainFilter
    compressor = LLMChainExtractor.from_llm(llm)

"""

In [ ]:
"""
            - Taxonomy of Document Compressors

DOCUMENT COMPRESSORS IN LANGCHAIN
├── 1. EXTRACTION-BASED (LLM)
│   └── LLMChainExtractor      → Extracts only relevant sentences per doc
│
├── 2. FILTERING-BASED (LLM & Embeddings)
│   ├── LLMChainFilter         → Binary yes/no: keeps or discards full doc
│   └── EmbeddingsFilter       → Drops docs below embedding similarity cutoff
│
├── 3. RE-RANKING-BASED (Cross-Encoders)
│   ├── CohereRerank           → SOTA cloud-based neural cross-encoder
│   ├── FlashrankRerank        → Ultra-fast local lightweight reranker
│   └── CrossEncoderReranker   → HuggingFace-based cross-encoders (BGE, etc.)
│
└── 4. PIPELINE COMPRESSORS
    └── DocumentCompressorPipeline → Chains multiple compressors in sequence



"""

### 3. Practical Examples of ContextualCompressionRetriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [ ]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


### 4. Complete Summary of ContextualCompressionRetriever

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│               CONTEXTUAL COMPRESSION RETRIEVER                   │
│                                                                  │
│  WHAT:  A wrapper retriever that retrieves documents first,      │
│         then filters, extracts, or re-ranks them for relevance.  │
│                                                                  │
│  WHY:   Base retrieval can return long/noisy documents.          │
│         Compression reduces tokens and improves answer focus.    │
│                                                                  │
│  WHERE: After a base retriever and before Prompt / LLM.          │
│                                                                  │
│  INTERNAL FLOW:                                                  │
│    Query → Base Retriever → Candidate Documents → Compressor     │
│          → Filtered / Extracted / Re-ranked Documents → LLM      │
│                                                                  │
│  BASIC SYNTAX:                                                   │
│    compression_retriever = ContextualCompressionRetriever(       │
│        base_retriever=base_retriever,                            │
│        base_compressor=compressor                                │
│    )                                                             │
│    docs = compression_retriever.invoke("your query")             │
│                                                                  │
│  COMMON COMPRESSORS:                                             │
│    LLMChainExtractor → Extract relevant sentences/passages       │
│    LLMChainFilter    → Keep or remove full documents             │
│    EmbeddingsFilter  → Remove weak semantic matches              │
│    CohereRerank      → High-quality API cross-encoder reranking  │
│    FlashrankRerank   → Lightweight/local reranking               │
│    CompressorPipeline→ Combine split + filter + rerank           │
│                                                                  │
│  KEY DESIGN PATTERNS:                                            │
│    Similarity Top-10 → EmbeddingsFilter → LLM                    │
│    MMR Top-20 → Reranker Top-4 → LLM                             │
│    MultiQuery → Merge → Compressor → LLM                         │
│                                                                  │
│  BEST FOR:                                                       │
│    ✅ Large PDFs, reports, manuals, knowledge bases              │
│    ✅ Token-sensitive RAG systems                                │
│    ✅ High precision customer-support or enterprise search       │
│    ✅ Removing irrelevant retrieved text                         │
│                                                                   │
│  LIMITATIONS:                                                     │
│    ❌ LLM extraction adds latency and cost                        │
│    ❌ Extractors can omit important context                       │
│    ❌ Rerankers improve ranking, but cannot find missed documents │
│    ❌ Exact wording may be unsuitable for legal/audit use         │
│                                                                   │
│  DEFAULT RECOMMENDATION:                                          │
│    Start with:                                                    │
│      Base Retriever: MMR, k=10-20                                 │
│      Compressor: EmbeddingsFilter or FlashrankRerank              │
│      Final output: top 3-5 documents                              │
│                                                                   │
│  GOLDEN RULE:                                                     │
│  "Retrieve broadly for recall, then compress or rerank for        │
│   precision. Do not send every retrieved chunk directly to        │
│   the LLM."                                                       │
└───────────────────────────────────────────────────────────────────┘


"""

In [ ]:
""" 
    Complete RAG architecture
    ===========================


                                RAG SYSTEM
        ┌──────────────────────────────────────────────────────────────┐
        │                                                              │
        │  User Question                                               │
        │       │                                                      │
        │       ▼                                                      │
        │  ┌──────────────┐                                            │
        │  │ Query        │                                            │
        │  └──────┬───────┘                                            │
        │         │                                                    │
        │         ▼                                                    │
        │  ┌───────────────────┐                                       │
        │  │ Base Retriever    │                                       │
        │  │                   │                                       │
        │  │ Vector Store      │                                       │
        │  │ / MMR / BM25      │                                       │
        │  └─────────┬─────────┘                                       │
        │            │                                                 │
        │            ▼                                                 │
        │  ┌───────────────────┐                                       │
        │  │ Retrieved Docs    │                                       │
        │  │                   │                                       │
        │  │ D1 D2 D3 D4 D5    │                                       │
        │  └─────────┬─────────┘                                       │
        │            │                                                 │
        │            ▼                                                 │
        │  ┌────────────────────────────┐                              │
        │  │ Contextual Compression     │                              │
        │  │                            │                              │
        │  │ Filter / Extract useful    │                              │
        │  │ information                │                              │
        │  └────────────┬───────────────┘                              │
        │               │                                              │
        │               ▼                                              │
        │  ┌────────────────────────────┐                              │
        │  │ Compressed Context         │                              │
        │  │                            │                              │
        │  │ Only relevant information  │                              │
        │  └────────────┬───────────────┘                              │
        │               │                                              │
        │               ▼                                              │
        │  ┌────────────────────────────┐                              │
        │  │ Prompt                     │                              │
        │  │                            │                              │
        │  │ Context + Question         │                              │
        │  └────────────┬───────────────┘                              │
        │               │                                              │
        │               ▼                                              │
        │  ┌────────────────────────────┐                              │
        │  │ LLM                        │                              │
        │  └────────────┬───────────────┘                              │
        │               │                                              │
        │               ▼                                              │
        │            ANSWER                                            │
        │                                                              │
        └──────────────────────────────────────────────────────────────┘



"""